# Prepare SimpleStories

Select the project's Python environment and run the cells in order with internet access. The notebook caches the published SimpleStories training split under `data/` and draws up to 60,000 training stories and 1,024 validation stories from it. The published test split is reserved for final evaluation. The tokenized corpus goes under `data/`.

In [ ]:
import json
import os
import sys
from itertools import islice
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from dotenv import load_dotenv

working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "src" / "settings.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from within the MiniReasoner project.")

load_dotenv(project_root / ".env")
from datasets import load_dataset

sys.path.insert(0, str(project_root))
from src.settings import DATASET_CACHE_DIR as cache_dir
from src.settings import SIMPLESTORIES_TRAIN_FILES
from src.settings import TOKENIZED_DATA_DIR as tokenized_path
from src.settings import VOCAB_SIZE
from src.tokenizer import EOS, TOKEN_DTYPE, pack_documents, sha256, train_tokenizer

dataset_name = "SimpleStories/SimpleStories"
source_splits = {"train": "train", "val": "train"}
story_limits = {"train": 60_000, "val": 1_024}
seed = 42

c:\Users\sampa\Dev\MiniReasoner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Select training and validation stories from the published training split

In [2]:
if min(story_limits.values()) < 1:
    raise ValueError("Story limits must be positive.")

token = os.environ.get("HF_TOKEN", "").strip() or None
dataset = load_dataset(
    dataset_name,
    data_files={"train": SIMPLESTORIES_TRAIN_FILES},
    split="train",
    cache_dir=str(cache_dir),
    # The Hub metadata describes both splits, but only train shards were requested.
    verification_mode="no_checks",
    token=token,
)

rows = dataset.select_columns(["story"]).shuffle(seed=seed)
cleaned = (row["story"].strip() for row in rows)
usable = (story for story in cleaned if story and EOS not in story)

validation = list(islice(usable, story_limits["val"]))
validation_texts = set(validation)
training = list(islice((story for story in usable if story not in validation_texts), story_limits["train"]))
stories = {"train": training, "val": validation}
for split, selected in stories.items():
    if not selected:
        raise ValueError(f"SimpleStories returned no usable stories for {split}.")
    print(f"Selected {len(selected):,} {split} stories.")

Generating train split: 100%|██████████| 2115696/2115696 [00:22<00:00, 92165.87 examples/s] 


Selected 60,000 train stories.
Selected 1,024 val stories.


## Inspect the corpus

In [3]:
for split, documents in stories.items():
    word_counts = [len(story.split()) for story in documents]
    p10, median, p90 = np.percentile(word_counts, [10, 50, 90])
    print(f"{split}: {len(documents):,} stories, {sum(word_counts):,} words")
    print(f"  Words per story: p10={p10:.0f}, median={median:.0f}, p90={p90:.0f}")

print("\nExample training story\n")
print(stories["train"][0])

train: 60,000 stories, 13,498,645 words
  Words per story: p10=109, median=208, p90=378
val: 1,024 stories, 235,117 words
  Words per story: p10=110, median=212, p90=385

Example training story

Swiftly, a boy gazed at the clock ticking on the wall. The time was short, and he felt the weight of the world on his shoulders. He had to finish his big drawing before the sun set. It was a magical world full of colors and shapes, but he felt stuck. The pencil felt heavy in his hand. 

He looked out the window, where the sky turned from blue to orange. The sun was his deadline. As it dipped lower, he thought about his plan. He wanted to make a picture that showed both light and dark. But now, all he saw was blank paper. He sighed, feeling like he would fail. 

Suddenly, a tiny voice whispered from the corner of the room. A little fairy appeared, her wings shimmering like stars. "Why do you worry?" she asked. "Just let the colors speak." The boy was surprised. He had never seen a fairy before. 

## Train the tokenizer

Train byte-level BPE on the training split. Individual digits stay separate for later arithmetic training.

In [4]:
tokenizer = train_tokenizer(stories["train"], VOCAB_SIZE)
preview = tokenizer.encode(stories["train"][0], add_special_tokens=False)

print(f"Vocabulary size: {tokenizer.get_vocab_size():,}")
print(f"First 25 tokens: {preview.tokens[:25]}")

Vocabulary size: 1,024
First 25 tokens: ['S', 'w', 'i', 'ft', 'ly', ',', 'Ġa', 'Ġboy', 'Ġg', 'a', 'z', 'ed', 'Ġat', 'Ġthe', 'Ġcl', 'o', 'ck', 'Ġt', 'ic', 'king', 'Ġon', 'Ġthe', 'Ġw', 'all', '.']


## Pack and save the corpus

Save the packed token files, tokenizer, and manifest in the chosen output directory. Each packed story ends with an EOS token. Existing packed files are protected.

In [5]:
tokenized_path.mkdir(parents=True, exist_ok=True)
output_files = ("train.bin", "val.bin", "tokenizer.json", "manifest.json")
for name in output_files:
    destination = tokenized_path / name
    if destination.exists():
        raise FileExistsError(f"Packed output already exists: {destination}")

with TemporaryDirectory(prefix=".prepare-", dir=tokenized_path) as temporary:
    staging = Path(temporary)
    tokenizer_path = staging / "tokenizer.json"
    tokenizer_path.write_text(tokenizer.to_str(pretty=True), encoding="utf-8")

    manifest = {
        "version": 1,
        "dtype": TOKEN_DTYPE.str,
        "seed": seed,
        "vocab_size": tokenizer.get_vocab_size(),
        "eos_id": tokenizer.token_to_id(EOS),
        "tokenizer_sha256": sha256(tokenizer_path),
        "dataset": dataset_name,
        "source_splits": source_splits,
    }
    for split, documents in stories.items():
        manifest[split] = pack_documents(staging / f"{split}.bin", tokenizer, iter(documents))

    (staging / "manifest.json").write_text(
        json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
    )
    # Publish the manifest last so the trainer sees only a complete corpus.
    published = []
    try:
        for name in output_files:
            destination = tokenized_path / name
            (staging / name).rename(destination)
            published.append(destination)
    except BaseException:
        for destination in published:
            destination.unlink(missing_ok=True)
        raise

for split in stories:
    print(f"{split}: {manifest[split]['tokens']:,} packed tokens")
print(f"Saved to {tokenized_path}")

train: 24,491,785 packed tokens
val: 425,276 packed tokens
Saved to c:\Users\sampa\Dev\MiniReasoner\data\tokenized


## Verify end-of-story tokens

Use the saved tokenizer and the selected stories to check every story boundary in `train.bin` and `val.bin`. Each file must also end with EOS.

In [6]:
from tokenizers import Tokenizer

saved_tokenizer = Tokenizer.from_file(str(tokenized_path / "tokenizer.json"))
saved_manifest = json.loads((tokenized_path / "manifest.json").read_text(encoding="utf-8"))
eos_id = saved_tokenizer.token_to_id(EOS)
if eos_id is None or eos_id != saved_manifest["eos_id"]:
    raise ValueError("The saved tokenizer and manifest disagree on the EOS token.")


In [7]:
for split in ("train", "val"):
    documents = stories[split]
    packed_path = tokenized_path / f"{split}.bin"
    split_info = saved_manifest[split]
    if not documents or len(documents) != split_info["documents"]:
        raise ValueError(f"{split}: selected stories do not match the saved document count.")
    if packed_path.stat().st_size != split_info["tokens"] * TOKEN_DTYPE.itemsize:
        raise ValueError(f"{split}: packed file size does not match the saved token count.")

    tokens = np.memmap(packed_path, dtype=TOKEN_DTYPE, mode="r")
    offset = 0
    for start in range(0, len(documents), 2048):
        batch = documents[start : start + 2048]
        encoded = saved_tokenizer.encode_batch(batch, add_special_tokens=False)
        lengths = [len(story.ids) + 1 for story in encoded]
        eos_positions = offset + np.cumsum(lengths) - 1
        if eos_positions[-1] >= len(tokens) or np.any(tokens[eos_positions] != eos_id):
            raise ValueError(f"{split}: missing EOS in the batch starting at story {start + 1:,}.")
        offset = int(eos_positions[-1]) + 1

    if offset != len(tokens) or np.count_nonzero(tokens == eos_id) != len(documents):
        raise ValueError(f"{split}: expected exactly one EOS per story and EOS at the end of the file.")
    print(f"{split}: verified EOS after all {len(documents):,} stories, including the final token.")
    del tokens

train: verified EOS after all 60,000 stories, including the final token.
val: verified EOS after all 1,024 stories, including the final token.
